
# GSM + GLEAM Closed-Loop Recovery (`gsm_recovery`)

This notebook adapts the closed-loop `newnucal` test to use a more realistic sky:

- diffuse emission from a Global Sky Model (GSM)
- compact emission from a GLEAM catalog
- smooth time/frequency gain perturbations
- radiometric complex visibility noise

The fit uses the existing `newnucal` infrastructure to recover sky, beam, and gain parameters and tracks the noise-weighted reduced chi-squared. The practical goal is to drive the fit to $\chi^2_\nu \approx 1$.

This notebook is written to be editable. The two most likely things you may need to adjust locally are:

1. the import path for the GSM package (`pygdsm` / `pygsm`)
2. the path and column names for the GLEAM catalog file


## 1. Imports and plotting setup

In [ ]:
import os
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import healpy as hp

from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u
from astropy.table import Table

jax.config.update("jax_enable_x64", False)

from newnucal import (
    HERAArray,
    BeamModel,
    ForwardModel,
    Calibrator,
    apply_gains,
    init_gain_params,
)
from newnucal.basis import BeamBasis, SkyBasis
from newnucal.sky import SkyModel
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

## 2. Configuration

In [ ]:
# --- Array / observing setup ---
hexnum = 4
sep_m = 14.6

nfreq = 64
freqs = np.linspace(50e6, 225e6, nfreq)  # Hz
freq_mhz = freqs / 1e6
dnu_hz = float(np.median(np.diff(freqs)))

ntime = 8
integration_time_s = 7.5 * 60.0  # 7.5 minutes per integration

hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
times = t0 + np.arange(ntime) * integration_time_s * u.s

# --- Sky / beam resolution and spectral smoothness ---
sky_nside = 32
beam_nside = 16
beam_eta_max = 20e-9
sky_eta_max = 60e-9

# --- Eigenbasis configuration ---
beam_diameters = [14.1, 14.6, 15.1]   # dish diameters (m) for beam ensemble
n_beam_modes = 17                      # SVD modes to retain in beam eigenbasis
n_sky_modes = 5                        # SVD modes to retain in sky eigenbasis

# --- Sky model ingredients ---
ref_freq_hz = 151e6
gleam_catalog_path = Path("GLEAM_EGC_v2.fits")   # edit if needed
gleam_flux_col_candidates = ["int_flux_151", "S_151", "Fint151", "peak_flux_151"]
gleam_alpha_col_candidates = ["alpha", "sp_index", "SpectralIndex"]
max_gleam_sources = 5000

# --- Noise model ---
T_rx_K = 100.0
aperture_efficiency = 0.7
noise_seed = 1234

# --- Joint fit configuration ---
fit_kwargs = dict(
    sky_step_size=0.9,
    sky_anderson_history=3,
    sky_aa_start=1,
    beam_anderson_history=2,
    beam_aa_start=1,
    beam_aa_damping=0.5,
    beam_aa_ridge=1e-8,
    beam_aa_max_weight=10,
    solve_every={
        'gains':         10,
        'beam':           1,
        'sky_max':        5,
        'beam_max':       5,
        'beam_lbfgs_max': 60,
    },
    beam_step_size=0.5,
    verbose=True,
)

target_reduced_chi2 = 1.05
max_fit_rounds = 4

## 3. Helper functions for GSM, GLEAM, and radiometric noise

In [ ]:

k_B = 1.380649e-23
c = 299792458.0

def load_gsm_temperature_maps(freqs_hz, nside):
    gsm = None
    try:
        from pygdsm import GlobalSkyModel2016
        gsm = GlobalSkyModel2016(freq_unit="Hz")
    except Exception:
        pass
    if gsm is None:
        try:
            from pygdsm import GlobalSkyModel
            gsm = GlobalSkyModel(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        try:
            from pygsm import GlobalSkyModel2016
            gsm = GlobalSkyModel2016(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        try:
            from pygsm import GlobalSkyModel
            gsm = GlobalSkyModel(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        raise ImportError(
            "Could not import a GSM model. Install pygdsm/pygsm or edit "
            "load_gsm_temperature_maps() for your local GSM package."
        )

    # Galactic -> Equatorial (Celestial/ICRS-like) rotator
    rot_gc = hp.Rotator(coord=["G", "C"])
    maps = []
    for nu in freqs_hz:
        try:
            m_gal = gsm.generate(float(nu))
        except TypeError:
            # some GSM packages expect MHz
            m_gal = gsm.generate(float(nu) / 1e6)
        m_gal = np.asarray(m_gal, dtype=np.float64)
        # Rotate from Galactic to Equatorial before any regridding
        m_eq = rot_gc.rotate_map_pixel(m_gal)
        if hp.get_nside(m_eq) != nside:
            m_eq = hp.ud_grade(m_eq, nside_out=nside, power=0)
        maps.append(m_eq.astype(np.float32))
    return np.stack(maps, axis=1)  # (npix, nfreq)

def _pick_column(tab, candidates):
    for name in candidates:
        if name in tab.colnames:
            return name
    return None

def load_gleam_sources(path, max_sources=None):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"GLEAM catalog not found at {path}. Point the notebook at a local "
            "GLEAM FITS/VOTable file."
        )

    tab = Table.read(path)
    ra_col = _pick_column(tab, ["RAJ2000", "ra", "RA", "RA_deg"])
    dec_col = _pick_column(tab, ["DEJ2000", "dec", "DEC", "DEC_deg"])
    flux_col = _pick_column(tab, gleam_flux_col_candidates)
    alpha_col = _pick_column(tab, gleam_alpha_col_candidates)
    if ra_col is None or dec_col is None or flux_col is None:
        raise ValueError(
            "Could not identify RA/DEC/151-MHz flux columns in the GLEAM catalog. "
            "Edit the candidate column lists near the top of the notebook."
        )

    ra_deg = np.asarray(tab[ra_col], dtype=np.float64)
    dec_deg = np.asarray(tab[dec_col], dtype=np.float64)
    s151_jy = np.asarray(tab[flux_col], dtype=np.float64)

    if alpha_col is None:
        alpha = -0.8 * np.ones_like(s151_jy)
    else:
        alpha = np.asarray(tab[alpha_col], dtype=np.float64)
        alpha[~np.isfinite(alpha)] = -0.8

    good = np.isfinite(ra_deg) & np.isfinite(dec_deg) & np.isfinite(s151_jy) & (s151_jy > 0)
    ra_deg, dec_deg, s151_jy, alpha = ra_deg[good], dec_deg[good], s151_jy[good], alpha[good]

    if max_sources is not None and len(s151_jy) > max_sources:
        keep = np.argsort(s151_jy)[-max_sources:]
        ra_deg, dec_deg, s151_jy, alpha = ra_deg[keep], dec_deg[keep], s151_jy[keep], alpha[keep]

    return dict(ra_deg=ra_deg, dec_deg=dec_deg, s151_jy=s151_jy, alpha=alpha)

def gleam_flux_cube_to_healpix(freqs_hz, nside, gleam, ref_freq_hz=151e6):
    npix = hp.nside2npix(nside)
    ra = np.asarray(gleam["ra_deg"])
    dec = np.asarray(gleam["dec_deg"])
    s151 = np.asarray(gleam["s151_jy"])
    alpha = np.asarray(gleam["alpha"])

    theta = np.deg2rad(90.0 - dec)
    phi = np.deg2rad(ra)
    pix = hp.ang2pix(nside, theta, phi)

    cube = np.zeros((npix, len(freqs_hz)), dtype=np.float32)
    scale = (freqs_hz[None, :] / ref_freq_hz) ** alpha[:, None]
    src_flux = s151[:, None] * scale
    for si, px in enumerate(pix):
        cube[px] += src_flux[si].astype(np.float32)
    return cube

def kelvin_to_jy_per_pix(temp_K, freqs_hz, nside):
    omega_pix = 4.0 * np.pi / hp.nside2npix(nside)
    prefac = 2.0 * k_B * (freqs_hz**2) / c**2 / 1e-26  # Jy / sr / K
    return temp_K * prefac[None, :] * omega_pix

def build_gsm_plus_gleam_flux_cube(freqs_hz, nside, gleam_path, max_sources=None):
    gsm_temp_K = load_gsm_temperature_maps(freqs_hz, nside)
    gsm_jy = kelvin_to_jy_per_pix(gsm_temp_K, freqs_hz, nside)
    gleam = load_gleam_sources(gleam_path, max_sources=max_sources)
    gleam_jy = gleam_flux_cube_to_healpix(freqs_hz, nside, gleam)
    return gsm_temp_K, gsm_jy, gleam, gleam_jy, gsm_jy + gleam_jy

def airy_collecting_area(diameter_m, aperture_eff=0.7):
    return aperture_eff * np.pi * (diameter_m / 2.0) ** 2

def approximate_radiometric_sigma_jy(freqs_hz, gsm_temp_K, dish_diameter_m, dnu_hz, tint_s,
                                     T_rx_K=100.0, aperture_eff=0.7):
    T_sky_f = np.mean(gsm_temp_K, axis=0)
    T_sys_f = T_sky_f + T_rx_K
    A_eff = airy_collecting_area(dish_diameter_m, aperture_eff=aperture_eff)
    sefd_jy = 2.0 * k_B * T_sys_f / A_eff / 1e-26
    sigma = sefd_jy / np.sqrt(2.0 * dnu_hz * tint_s)
    return sigma.astype(np.float32), T_sys_f.astype(np.float32)

def complex_gaussian_noise(shape, sigma_f, rng):
    sigma = np.asarray(sigma_f, dtype=np.float32)[None, :, None]
    nre = rng.standard_normal(shape).astype(np.float32)
    nim = rng.standard_normal(shape).astype(np.float32)
    return sigma * (nre + 1j * nim) / np.sqrt(2.0)


## 4. Array, beam, times, and rotation matrices

In [ ]:
array = HERAArray.from_hex(hexnum=hexnum, sep=sep_m)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")

beam_model = BeamModel(nside=beam_nside, freqs=freqs,
                       basis=BeamBasis.from_dpss(freqs, beam_eta_max))
print(f"Beam DPSS modes: {beam_model.A.shape[1]}")

## 5. Build the GSM + GLEAM sky cube and project to the sky DPSS basis

In [ ]:
gsm_temp_K, gsm_jy, gleam, gleam_jy, flux_true = build_gsm_plus_gleam_flux_cube(
    freqs,
    sky_nside,
    gleam_catalog_path,
    max_sources=max_gleam_sources,
)

print(f"GSM temperature cube shape: {gsm_temp_K.shape}")
print(f"GLEAM source count used:    {len(gleam['s151_jy'])}")
print(f"Combined sky flux cube:     {flux_true.shape}  [Jy / pixel]")

sky_model = SkyModel(nside=sky_nside, freqs=freqs,
                     basis=SkyBasis.from_dpss(freqs, sky_eta_max))
sky_coeffs_true = jnp.array(sky_model.basis.project(flux_true), dtype=jnp.float32)

print(f"Sky DPSS modes:             {sky_model.A.shape[1]}")
print(f"sky_coeffs_true shape:      {sky_coeffs_true.shape}")

In [ ]:
# Sky eigenbasis: each HEALPix pixel's GSM+GLEAM spectrum is one ensemble sample
sky_basis = SkyBasis.from_ensemble(freqs, flux_true, n_modes=n_sky_modes)
print(f"Sky eigenbasis:  {sky_basis.nmodes} modes  ({sky_basis.nfreq} freqs)")
print(f"  top svals: {sky_basis.svd_svals[:8]}")

# Beam eigenbasis: above-horizon power spectra from Airy beams at several diameters
beam_basis = BeamBasis.from_beam_diameters(
    freqs,
    diameters=beam_diameters,
    nside=beam_nside,
    n_modes=n_beam_modes,
)
print(f"Beam eigenbasis: {beam_basis.nmodes} modes  ({beam_basis.nfreq} freqs)")
print(f"  top svals: {beam_basis.svd_svals[:8]}")

In [ ]:
# Joint sky×beam product basis: outer product of top SVD modes, then SVD
# Use the raw (pre-QR) modes from each basis with their singular-value weights.
Vt_bm = np.vstack([beam_basis.svd_mean[None, :], beam_basis.svd_modes])
S_bm  = np.concatenate([[float(beam_basis.n_samples)], beam_basis.svd_svals])

Vt_fl = np.vstack([sky_basis.svd_mean[None, :], sky_basis.svd_modes])
S_fl  = np.concatenate([[float(sky_basis.n_samples)], sky_basis.svd_svals])

outer = (S_bm[:n_beam_modes + 1, None, None] * Vt_bm[:n_beam_modes + 1, None, :]
       * S_fl[None, :n_sky_modes + 1, None] * Vt_fl[None, :n_sky_modes + 1, :])
outer = outer.reshape(-1, nfreq)
outer_mean = outer.mean(axis=0, keepdims=True)
_, S_prod, Vt_prod = np.linalg.svd(outer - outer_mean)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(beam_basis.svd_svals / beam_basis.svd_svals[0], '.-', label='beam')
ax.semilogy(sky_basis.svd_svals / sky_basis.svd_svals[0], '.-', label='sky')
ax.semilogy(S_prod / S_prod[0], '.-', label='product')
ax.set_ylim(1e-10, 1)
ax.set_xlabel('SVD mode index')
ax.set_ylabel('Relative singular value')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
sky_basis.save('sky_basis_gsm_gleam.npz')
beam_basis.save('beam_basis_airy.npz')
print(f"Saved sky eigenbasis  ({sky_basis.nmodes} modes) → sky_basis_gsm_gleam.npz")
print(f"Saved beam eigenbasis ({beam_basis.nmodes} modes) → beam_basis_airy.npz")